# Gradient and Relaxation Analysis - Data-Like Events
## Using WC_loss with data-like events from ROOT files

In [1]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

import sys
sys.path.append('..')

from tools.geometry import generate_detector
from tools.utils import load_single_event, save_single_event, generate_random_params, print_particle_params
from tools.losses import WC_loss
from tools.simulation import setup_event_simulator
from tools.generate import read_photon_data_from_photonsim

import jax
import jax.numpy as jnp
import time

from jax import jit
from pathlib import Path

from matplotlib import pyplot as plt
plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10

import numpy as np
from functools import partial
import pickle
from tqdm import tqdm
from jax import grad, jit, vmap, value_and_grad
import uproot

## Setup Detector and Event Simulators

In [ ]:
default_json_filename = '../config/SK_geom_config.json'
#data_file = '../data/water/muon/50_data_like_events.root'
data_file = '../data/water/muon/muon_gun_1050_MeV_100_events_fixed_energy.root'

detector = generate_detector(default_json_filename)
detector_points = jnp.array(detector.all_points)
detector_radius = detector.S_radius
NUM_DETECTORS = len(detector_points)
Nphot = 3_000_000

# Define temperatures for analysis
temperatures =  [0.0, 0.2]
K=6
# Setup prediction simulators for different temperatures (is_data=False)
prediction_simulators = {temperature: setup_event_simulator(default_json_filename, Nphot, temperature, K=K, is_data=False) 
                        for temperature in temperatures}

# Setup data simulator for generating the target event (is_data=True, temperature=0.0)
data_simulator = setup_event_simulator(default_json_filename, Nphot, temperature=0.0, K=K,
                                      is_data=True, is_calibration=False)

print(f"Number of detectors: {NUM_DETECTORS}")
print(f"Temperatures for analysis: {temperatures}")

# Load ROOT file information
with uproot.open(data_file) as file:
    tree = file['OpticalPhotons']
    n_entries = tree.num_entries
print(f"ROOT file has {n_entries} entries")

## Generate Test Data-Like Event

In [ ]:
# Generate random entry index and event parameters
key = jax.random.PRNGKey(719007)
entry_key = jax.random.PRNGKey(12345)

# Select random entry from ROOT file
entry_idx = 4#int(jax.random.randint(entry_key, shape=(), minval=0, maxval=n_entries))
print(f"Loading entry {entry_idx} from ROOT file")

# Load photon data from ROOT file
photon_data = read_photon_data_from_photonsim(data_file, entry_idx)
photon_data['N'] = len(photon_data['photon_origins'])

# Generate track parameters (position and direction)
# Use detector bounds for random position generation
detector_bounds = {
    'type': 'cylinder',
    'r': detector.r,
    'H': detector.H
}

# fraction = 0.6
# r_vert = jax.random.uniform(key, shape=(), minval=0, maxval=detector_bounds['r'] * fraction)
# key, _ = jax.random.split(key)
# theta = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
# key, _ = jax.random.split(key)
# z_vert = jax.random.uniform(key, shape=(), minval=-detector_bounds['H']/2 * fraction, 
#                            maxval=detector_bounds['H']/2 * fraction)
# true_position = jnp.array([r_vert * jnp.cos(theta), r_vert * jnp.sin(theta), z_vert])

true_position = jnp.array([+10., 0., 0.])

# Random direction
# key, _ = jax.random.split(key)
# phi = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
# key, _ = jax.random.split(key)
# cos_theta = jax.random.uniform(key, shape=(), minval=-1, maxval=1)
# sin_theta = jnp.sqrt(1 - cos_theta**2)
# true_direction = jnp.array([sin_theta * jnp.cos(phi), sin_theta * jnp.sin(phi), cos_theta])

true_direction = jnp.array([1., 0., 0.])

# Use energy from ROOT file
true_energy = photon_data['energy']

# Detector parameters
detector_params = (
    jnp.array(50.),           # scatter_length
    jnp.array(0.2),          # reflection_rate
    jnp.array(50.),           # absorption_length
    jnp.array(0.001)         # gumbel_softmax_temp
)

# Create particle parameters tuple
true_params = (true_energy, true_position, true_direction)

# Generate data-like event using data simulator with temperature=0.0
#true_data_clean = jax.lax.stop_gradient(data_simulator(true_params, detector_params, key, photon_data))
true_data = jax.lax.stop_gradient(data_simulator(true_params, detector_params, key, photon_data))

theta = jnp.arccos(jnp.clip(true_direction[2], -1.0, 1.0))
phi = jnp.arctan2(true_direction[1], true_direction[0])
spherical_params = (true_energy, true_position, jnp.array([theta, phi]))
pred_data = jax.lax.stop_gradient(prediction_simulators[0.2](spherical_params, detector_params, key))

print("True parameters:")
print(f"  Energy: {true_energy:.2f} MeV")
print(f"  Position: [{true_position[0]:.2f}, {true_position[1]:.2f}, {true_position[2]:.2f}] m")
print(f"  Direction: [{true_direction[0]:.3f}, {true_direction[1]:.3f}, {true_direction[2]:.3f}]")
print(f"  Photon data: {photon_data['N']} photons")

In [ ]:
# ## Noise Functions

# def add_poisson_noise(pe_values, key):
#     """
#     Add Poisson fluctuations to PE values
    
#     Args:
#         pe_values: array of expected PE counts
#         key: JAX random key
    
#     Returns:
#         observed_pe: Poisson-sampled PE counts
#     """
#     return jax.random.poisson(key, pe_values)

# def add_timing_noise(pe_values, times, key, tts=0.2):
#     """
#     Add transit time spread (TTS) noise to timing measurements.
#     Noise is scaled by 1/sqrt(PE) for multi-PE hits.
#     """
#     noise = jax.random.normal(key, shape=times.shape) * tts
    
#     # Scale noise by 1/sqrt(PE), cap at single PE
#     scaling = jnp.where(
#         pe_values >= 1.0,
#         1.0 / jnp.sqrt(pe_values),
#         1.0
#     )
    
#     # Only add noise where PE > 0
#     noisy_times = jnp.where(
#         pe_values > 0,
#         times + noise * scaling,
#         times
#     )
    
#     return noisy_times

# def add_detector_noise(charge_time_tuple, key, tts=0.2):
#     """
#     Add noise to both charge (PE) and timing
    
#     Args:
#         charge_time_tuple: (charge_array, time_array)
#         key: JAX random key
#         tts: transit time spread in ns
    
#     Returns:
#         (noisy_charge, noisy_time): tuple of arrays with noise added
#     """
#     charges, times = charge_time_tuple
    
#     # Split key for two operations
#     key1, key2 = jax.random.split(key)
    
#     # Add Poisson noise to charges
#     noisy_charges = add_poisson_noise(charges, key1)
    
#     # Add timing noise (using noisy charges for scaling)
#     noisy_times = add_timing_noise(noisy_charges, times, key2, tts)
    
#     return (noisy_charges, noisy_times)

# # Add noise to the clean data-like event
# noise_key = jax.random.PRNGKey(12345)
# true_data = add_detector_noise(true_data_clean, noise_key, tts=0.2)

# print(f"\nNoise added to data-like event:")
# print(f"  Original hits: {jnp.sum(true_data_clean[0] > 0)}")
# print(f"  Noisy hits: {jnp.sum(true_data[0] > 0)}")
# print(f"  Original total PE: {jnp.sum(true_data_clean[0]):.1f}")
# print(f"  Noisy total PE: {jnp.sum(true_data[0]):.1f}")

In [ ]:
from tools.visualization import create_detector_display

event_location = '../events/data_like_event.h5'

figures_dir = Path('figures')
figures_dir.mkdir(parents=True, exist_ok=True)

# Save the noisy data-like event
# Convert direction to spherical coordinates for saving
theta_sph = jnp.arccos(jnp.clip(true_direction[2], -1.0, 1.0))
phi_sph = jnp.arctan2(true_direction[1], true_direction[0])
true_params_spherical = (true_energy, true_position, jnp.array([theta_sph, phi_sph]))

save_single_event(true_data, true_params_spherical, detector_params, filename=event_location, calibration_mode=False)

loaded_trk_params, loaded_detector_params, loaded_indices_t, loaded_charges, loaded_times_t = load_single_event(event_location, NUM_DETECTORS, calibration_mode=False)
print_particle_params(loaded_trk_params)

detector_display = create_detector_display(default_json_filename)
detector_display(loaded_indices_t, loaded_charges, loaded_times_t, file_name='figures/data_event_display_charge.png', plot_time=False, log_scale=True)
detector_display(loaded_indices_t, loaded_charges, loaded_times_t, file_name='figures/data_event_display_time.png', plot_time=True, perc_min=0.0, perc_max=50.0)

pred_evt_location = '../events/pred_like_event.h5'
save_single_event(pred_data, true_params_spherical, detector_params, filename=pred_evt_location, calibration_mode=False)

loaded_trk_params, loaded_detector_params, loaded_indices, loaded_charges, loaded_times_p = load_single_event(pred_evt_location, NUM_DETECTORS, calibration_mode=False)
print_particle_params(loaded_trk_params)

detector_display = create_detector_display(default_json_filename)
detector_display(loaded_indices, loaded_charges, loaded_times_p, file_name='figures/pred_event_display_charge.png', plot_time=False, log_scale=True)
detector_display(loaded_indices, loaded_charges, loaded_times_p, file_name='figures/pred_event_display_time.png', plot_time=True, perc_min=0.0, perc_max=60.0)

formatted_times = jnp.zeros(len(loaded_times_p)).at[loaded_indices_t].set(loaded_times_t)
print(len(loaded_times_p))
detector_display(loaded_indices, loaded_charges, abs(formatted_times-loaded_times_p), file_name='figures/pred_event_display_time.png', plot_time=True, perc_min=0.0, perc_max=60.0)

## Hyperparameter Analysis Helper Functions

In [ ]:
# Define parameter min/max ranges based on detector geometry
param_min_max = (
    (0.0001, 1000.0),                    # Energy in MeV (min, max)
    (-detector.r, detector.r),           # Position X in meters (min, max)
    (-detector.r, detector.r),           # Position Y in meters (min, max)
    (-detector.H/2, detector.H/2),       # Position Z in meters (min, max)
    (0, 3.14159),                         # Theta (polar angle) in radians: 0 to π
    (0, 6.28318)                          # Phi (azimuthal angle) in radians: 0 to 2π
)

# Calculate parameter ranges (max - min) for gradient scaling
param_range_scales = jnp.array([
    param_min_max[0][1] - param_min_max[0][0],  # Energy range
    param_min_max[1][1] - param_min_max[1][0],  # Position X range
    param_min_max[2][1] - param_min_max[2][0],  # Position Y range (not used in analysis)
    param_min_max[3][1] - param_min_max[3][0],  # Position Z range (not used in analysis)
    param_min_max[4][1] - param_min_max[4][0],  # Theta range
    param_min_max[5][1] - param_min_max[5][0]   # Phi range
])

# Define parameter changes for analysis
param_changes = (
    jnp.array(200.0),             # Energy in MeV
    jnp.array([1.0, 0.0, 0.0]),   # position
    jnp.array([0.3, 0.3])         # direction (theta, phi)
)

def generate_param_ranges(particle_params, param_changes, num_points=121):
    """Generate parameter ranges for analysis"""
    param_ranges = []
    
    # Energy
    start = max(particle_params[0] - param_changes[0], 0.0001)
    end = particle_params[0] + param_changes[0]
    param_ranges.append(jnp.linspace(start, end, num_points))
    
    # Position X
    start = particle_params[1][0] - param_changes[1][0]
    end = particle_params[1][0] + param_changes[1][0]
    param_ranges.append(jnp.linspace(start, end, num_points))
    
    # Direction angles from true_direction
    true_theta = jnp.arccos(jnp.clip(particle_params[2][2], -1.0, 1.0))
    true_phi = jnp.arctan2(particle_params[2][1], particle_params[2][0])
    
    # Theta
    start = true_theta - param_changes[2][0]
    end = true_theta + param_changes[2][0]
    param_ranges.append(jnp.linspace(start, end, num_points))
    
    # Phi
    start = true_phi - param_changes[2][1]
    end = true_phi + param_changes[2][1]
    param_ranges.append(jnp.linspace(start, end, num_points))
    
    return param_ranges

def generate_plot_data_wc_prediction(param_index, param_values, simulator, particle_params,
                                    detector_params, key, true_data, detector_points,
                                    lambda_poisson=1.0, lambda_time=1.0):
    """Generate loss and gradient data using WC_loss with prediction simulators against data-like events"""
    losses = []
    gradients = []
    
    @jit
    def loss_and_grad_fn(p_params, d_params):
        def loss_fn(p):
            # Convert direction to spherical angles for prediction simulator
            energy, position, direction = p
            theta = jnp.arccos(jnp.clip(direction[2], -1.0, 1.0))
            phi = jnp.arctan2(direction[1], direction[0])
            spherical_params = (energy, position, jnp.array([theta, phi]))
            
            simulated_data = simulator(spherical_params, d_params, key)
            return WC_loss(
                detector_points, *true_data, *simulated_data,
                lambda_poisson=lambda_poisson,
                lambda_time=lambda_time
            )
        return value_and_grad(loss_fn)(p_params)
    
    # Get the appropriate parameter range scale
    param_scale_indices = [0, 1, 4, 5]  # Energy, X, Theta, Phi
    param_scale = param_range_scales[param_scale_indices[param_index]]
    
    for new_value in param_values:
        # Modify particle parameter
        new_p_params = list(particle_params)
        
        if param_index == 0:  # Energy
            new_p_params[0] = new_value
        elif param_index == 1:  # Position X
            new_p_params[1] = new_p_params[1].at[0].set(new_value)
        elif param_index == 2:  # Theta
            # Convert theta back to cartesian direction
            theta = new_value
            phi = jnp.arctan2(particle_params[2][1], particle_params[2][0])
            new_direction = jnp.array([
                jnp.sin(theta) * jnp.cos(phi),
                jnp.sin(theta) * jnp.sin(phi),
                jnp.cos(theta)
            ])
            new_p_params[2] = new_direction
        elif param_index == 3:  # Phi
            # Convert phi back to cartesian direction
            theta = jnp.arccos(jnp.clip(particle_params[2][2], -1.0, 1.0))
            phi = new_value
            new_direction = jnp.array([
                jnp.sin(theta) * jnp.cos(phi),
                jnp.sin(theta) * jnp.sin(phi),
                jnp.cos(theta)
            ])
            new_p_params[2] = new_direction
        
        new_p_params = tuple(new_p_params)
        
        # Calculate loss and gradient
        loss, grad_val = loss_and_grad_fn(new_p_params, detector_params)
        
        # Extract gradient for the specific parameter
        if param_index == 0:  # Energy
            gradient = grad_val[0]
        elif param_index == 1:  # Position X
            gradient = grad_val[1][0]
        elif param_index == 2:  # Theta (need chain rule)
            # grad w.r.t. direction * d(direction)/d(theta)
            dir_grad = grad_val[2]
            theta = jnp.arccos(jnp.clip(particle_params[2][2], -1.0, 1.0))
            phi = jnp.arctan2(particle_params[2][1], particle_params[2][0])
            # Derivative of direction w.r.t. theta
            dd_dtheta = jnp.array([
                jnp.cos(theta) * jnp.cos(phi),
                jnp.cos(theta) * jnp.sin(phi),
                -jnp.sin(theta)
            ])
            gradient = jnp.dot(dir_grad, dd_dtheta)
        elif param_index == 3:  # Phi (need chain rule)
            # grad w.r.t. direction * d(direction)/d(phi)
            dir_grad = grad_val[2]
            theta = jnp.arccos(jnp.clip(particle_params[2][2], -1.0, 1.0))
            phi = jnp.arctan2(particle_params[2][1], particle_params[2][0])
            # Derivative of direction w.r.t. phi
            dd_dphi = jnp.array([
                -jnp.sin(theta) * jnp.sin(phi),
                jnp.sin(theta) * jnp.cos(phi),
                0.0
            ])
            gradient = jnp.dot(dir_grad, dd_dphi)
        
        # Multiply gradient by parameter range
        gradient = gradient * param_scale
        
        losses.append(loss)
        gradients.append(gradient)
    
    return jnp.array(losses), jnp.array(gradients)

## Temperature Gradient Analysis with Data-Like Events

In [ ]:
def calculate_numerical_gradient(losses, param_values):
    """Calculate numerical gradient using 5-point central difference for better accuracy"""
    gradients = np.zeros(len(losses))
    h = param_values[1] - param_values[0]  # Assuming uniform spacing

    for i in range(len(losses)):
        if i < 2:
            # Use forward difference for first two points
            if i == 0:
                gradients[i] = (-3*losses[0] + 4*losses[1] - losses[2]) / (2*h)
            else:  # i == 1
                gradients[i] = (losses[2] - losses[0]) / (2*h)
        elif i >= len(losses) - 2:
            # Use backward difference for last two points
            if i == len(losses) - 1:
                gradients[i] = (losses[-3] - 4*losses[-2] + 3*losses[-1]) / (2*h)
            else:  # i == len(losses) - 2
                gradients[i] = (losses[-1] - losses[-3]) / (2*h)
        else:
            # Use 5-point central difference for interior points
            gradients[i] = (-losses[i+2] + 8*losses[i+1] - 8*losses[i-1] + losses[i-2]) / (12*h)

    return gradients

def scale_gradients_by_percentile(gradients, percentile=99):
    """Scale gradients by their percentile value"""
    abs_gradients = np.abs(gradients)
    scale_factor = np.percentile(abs_gradients, percentile)
    if scale_factor > 1e-10:  # Avoid division by very small numbers
        return gradients / scale_factor
    else:
        return gradients

In [ ]:
# Generate parameter ranges
num_points = 31
param_ranges = generate_param_ranges(true_params, param_changes, num_points)
param_names = ['Energy', 'Position X', 'Theta', 'Phi']

# Temperature analysis with lambda_poisson=1 using prediction simulators against data-like event
print("Running Temperature Gradient Analysis with prediction simulators against data-like event...")

# Store all results
temperature_analysis = {
    'param_ranges': param_ranges,
    'param_names': param_names,
    'temperatures': temperatures,
    'particle_params': true_params,
    'case': 'temperature_gradient_data_vs_prediction',
    'results': {}
}

# Process each temperature using prediction simulators
for temp in temperatures:
    simulator = prediction_simulators[temp]
    temp_key = f'temp_{temp}'
    temperature_analysis['results'][temp_key] = {}
    
    print(f"\nProcessing temperature {temp} (prediction simulator vs data-like event)...")
    
    for param_idx, param_name in enumerate(tqdm(param_names, desc=f"T={temp}")):
        param_values = param_ranges[param_idx]
        
        losses, gradients = generate_plot_data_wc_prediction(
            param_idx, param_values, simulator,
            true_params, detector_params, key, true_data, detector_points,
            lambda_poisson=1.0, lambda_time=1.0
        )
        
        temperature_analysis['results'][temp_key][param_name] = {
            'losses': losses,
            'gradients': gradients
        }

# Calculate numerical gradients for T=0.0
print("\nCalculating numerical gradients for T=0.0...")
temp_0_key = 'temp_0.0'
for param_name in param_names:
    losses = np.array(temperature_analysis['results'][temp_0_key][param_name]['losses'])
    param_idx = param_names.index(param_name)
    param_values = np.array(param_ranges[param_idx])
    
    # Calculate numerical gradient (raw, no scaling)
    numerical_grad = calculate_numerical_gradient(losses, param_values)
    
    # Apply parameter range scaling to match analytical gradients
    param_scale_indices = [0, 1, 4, 5]  # Energy, X, Theta, Phi
    param_scale = param_range_scales[param_scale_indices[param_idx]]
    numerical_grad *= param_scale
    
    # Store numerical gradient
    temperature_analysis['results'][temp_0_key][param_name]['numerical_gradients'] = numerical_grad

# Scale all gradients by their 99th percentile for each temperature and parameter
print("\nScaling gradients by 99th percentile...")
for temp_key in temperature_analysis['results'].keys():
    for param_name in param_names:
        # Scale analytical gradients
        gradients = np.array(temperature_analysis['results'][temp_key][param_name]['gradients'])
        scaled_gradients = scale_gradients_by_percentile(gradients, percentile=99)
        temperature_analysis['results'][temp_key][param_name]['scaled_gradients'] = scaled_gradients
        
        # Scale numerical gradients if they exist
        if 'numerical_gradients' in temperature_analysis['results'][temp_key][param_name]:
            num_gradients = np.array(temperature_analysis['results'][temp_key][param_name]['numerical_gradients'])
            scaled_num_gradients = scale_gradients_by_percentile(num_gradients, percentile=99)
            temperature_analysis['results'][temp_key][param_name]['scaled_numerical_gradients'] = scaled_num_gradients

In [ ]:
# Plot temperature gradient analysis for data-like events
print("\nGenerating temperature gradient analysis plots for prediction vs data-like event...")

# Color palette
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

# Figure size appropriate for paper
fig, axs = plt.subplots(2, 4, figsize=(16, 6.5))

for param_idx, param_name in enumerate(param_names):
    ax_loss = axs[0, param_idx]
    ax_grad = axs[1, param_idx]
    
    param_values = param_ranges[param_idx]
    
    # Determine true value
    if param_idx == 0:  # Energy
        true_value = true_params[0]
    elif param_idx == 1:  # Position X
        true_value = true_params[1][0]
    elif param_idx == 2:  # Theta
        true_value = jnp.arccos(jnp.clip(true_params[2][2], -1.0, 1.0))
    elif param_idx == 3:  # Phi
        true_value = jnp.arctan2(true_params[2][1], true_params[2][0])
    
    # Plot for each temperature
    for temp_idx, temp in enumerate(temperatures):
        temp_key = f'temp_{temp}'
        results = temperature_analysis['results'][temp_key][param_name]
        
        # Create label with sigma symbol
        if temp == 0.0:
            label = 'Discrete AD'
        else:
            label = f'σ = {temp}r AD'
        color = colors[temp_idx]
        
        # Plot losses
        ax_loss.plot(param_values, results['losses'], 
                    color=color, lw=2.0, label=label)
        
        # Plot scaled analytical gradients
        ax_grad.plot(param_values, results['scaled_gradients'], 
                    color=color, lw=2.0, label=label)
        
        # Plot scaled numerical gradient for T=0.0
        if temp == 0.0 and 'scaled_numerical_gradients' in results:
            ax_grad.plot(param_values, results['scaled_numerical_gradients'], 
                       color='#8B0000', lw=2.0, linestyle='-', alpha=0.9,
                       label='Numerical')
    
    # Add reference lines
    ax_loss.axvline(x=true_value, color='black', linestyle='--', lw=1.5, alpha=0.7, 
                   label='True Value' if param_idx == 0 else '')
    ax_grad.axvline(x=true_value, color='black', linestyle='--', lw=1.5, alpha=0.7)
    ax_grad.axhline(y=0, color='gray', linestyle=':', lw=1.5, alpha=0.7, 
                   label='Zero Gradient' if param_idx == 0 else '')
    
    # Configure axes
    ax_loss.set_title(f'{param_name}', fontsize=14)
    
    # Set x-labels
    if param_name == 'Energy':
        xlabel = f'{param_name} (MeV)'
    elif param_name == 'Position X':
        xlabel = f'{param_name} (m)'
    else:  # Theta, Phi
        xlabel = f'{param_name} (rad)'
    
    ax_loss.set_xlabel(xlabel, fontsize=12)
    ax_grad.set_xlabel(xlabel, fontsize=12)
    
    # Y-labels for leftmost plots
    if param_idx == 0:
        ax_loss.set_ylabel('Loss', fontsize=14)
        ax_grad.set_ylabel('Gradient / Max(|Gradient|)', fontsize=14)
        ax_loss.legend(fontsize=10, loc='best')
        ax_grad.legend(fontsize=10, loc='best')
    
    ax_loss.grid(True, alpha=0.3)
    ax_grad.grid(True, alpha=0.3)

# Use mathcal for the title
fig.suptitle(r'Prediction vs Data-Like Events ($\mathcal{L}_{\mathrm{poisson}}$)', 
             fontsize=16, y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])

# Save figure
plt.savefig('../figures/temperature_gradient_analysis_data_vs_prediction.pdf', dpi=300, bbox_inches='tight')
plt.show()

print("\nAnalysis complete!")
print(f"Temperatures analyzed: {temperatures}")
print("Prediction simulators fitted to data-like event with gradients scaled by 99th percentile")

## Save Results

In [ ]:
# Save analysis results
with open('../output/temperature_analysis_data_vs_prediction.pkl', 'wb') as f:
    pickle.dump(temperature_analysis, f)

print("Results saved to ../output/temperature_analysis_data_vs_prediction.pkl")